# Enrich existing IBM Quantum job metadata

This notebook performs **read-only retrieval** for the IBM Quantum identifiers already recorded under `results/`. The repository contains both direct IBM Compute/Runtime jobs and Q-CTRL/QESEM Qiskit Function jobs; function jobs are also expanded into their associated Runtime jobs and sessions when IBM exposes them. It does not import a Sampler or Estimator and contains no `.run()` call, so it cannot submit a new quantum workload.

IBM documents usage as the time a QPU is locked to execute a workload. The calls used here only read existing batch/job records and therefore do not consume new QPU time. They can still be subject to normal API permissions, rate limits, and data-retention rules.

References: [workload usage](https://quantum.cloud.ibm.com/docs/en/guides/estimate-job-run-time), [job metrics](https://quantum.cloud.ibm.com/docs/en/api/qiskit-runtime-rest/tags/jobs), and [monitor/retrieve jobs](https://quantum.cloud.ibm.com/docs/en/guides/monitor-job).


## 1. Choose what to retrieve

The identifiers below were extracted and deduplicated from the JSON files currently under `results/`. The files contain **64 Runtime job IDs**, **50 Qiskit Function job IDs**, and **no batch/session IDs**. They are grouped by source directory for auditability. The separate testing batch is left commented out because it is not part of the paper data.


In [1]:
from pathlib import Path

def parse_ids(text):
    lines_without_comments = [line.split("#", 1)[0] for line in text.splitlines()]
    return list(dict.fromkeys(
        value.strip()
        for value in ",".join(lines_without_comments).split(",")
        if value.strip()
    ))

RUNTIME_JOB_IDS_TEXT = """
# results/ghz_parity
d8qvms6gbcrc73f3v89g, d8r53deab0ds73dr8phg, d8rf4dmab0ds73drk25g, d8ri28i01fac73d4c7v0,
# results/sampler_benchmark
d97cvqd2su3c739hvcf0, d96vk7sqp3as739r3t1g, d97f5iaf47jc73a6m8a0, d96ql9if47jc73a5tqtg, d96rhjgtcv6s73dk4if0, d96rta4qp3as739qvb6g, d979bokqp3as739rg9fg, d96thj8tcv6s73dk6o7g, d96uof0tcv6s73dk88hg, d97bctif47jc73a6i6h0, d96v222f47jc73a62o8g, d97eh3if47jc73a6lhcg, d96qk44qp3as739qu2mg, d96r6u4qp3as739qukng, d96rsj4qp3as739qvabg, d978e4kqp3as739rf8pg, d96stogtcv6s73dk60h0, d96tkft2su3c739hcac0,
# results/sampler_benchmark-QCTRL-RND
d9a036gtcv6s73dnrudg, d9a030l2su3c739l0asg, d9a032otcv6s73dnru8g, d9a034d2su3c739l0b20, d9a036cqp3as739ulvd0, d9a030d2su3c739l0as0, d9a032l2su3c739l0b00, d9a0348tcv6s73dnrua0,
# results/tfim_probe
d91h10j57qjs73b6bfng, d91irt7qq29s738np1i0, d9213luu9n7c73aniku0, d91i5fnccmks73d56rbg, d91issr57qjs73b6et40, d921d3vqq29s738ohgqg,
# results/tfim_probe-10K-8sites
d8skudtposuc738nfilg, d8t1ahlposuc738o14dg, d8tt4ddposuc738p9e4g, d8tuvn5posuc738pbs2g, d8u3t5stqbtc73d17b0g, d8s47u6ab0ds73dsbq4g, d8sdbcmkodhs7385eahg, d8seq9sbp3hs73830o90, d8sfutkbp3hs73832ceg, d8sl6jlposuc738nfum0, d8t25u5posuc738o2ehg, d8tttalbh0os73eq6ejg, d8u0i0lposuc738pdsv0, d8u56t4tqbtc73d196m0, d8s4ova01fac73d51on0, d8sdopa01fac73d5ca8g, d8serhktqbtc73curps0, d8sg00tbh0os73eo4mp0,
# results/tfim_probe-16layers-full-chain
d8vvclopknjs73a1nf4g, d902n1emvj5c73eimnj0, d8vmvpemvj5c73ei5140, d8vq2qo6c68s73ah66f0, d8vtd406c68s73ahb2fg, d900e8propqc738d4t90, d903vd1ropqc738da780, d8vnrm6mvj5c73ei6jt0, d8vr5ggpknjs73a1h7og, d8vtocopknjs73a1l2og,
"""

FUNCTION_JOB_IDS_TEXT = """
# results/ghz_parity
e58d4f4a-0560-4438-ad2c-407532b445fa, 2d258c03-4265-4ee9-b1a8-b974a14e047e, db75adbd-1537-4e7b-8d57-1e6839d2c08f, 3adfcd16-fe3a-4c8a-a36d-57bbb66ef6eb, df40092a-7e1c-41ab-9a52-db53914c9c70, 518b77c3-3b69-4c60-b390-9308f132a927,
# results/sampler_benchmark
f0d2caaa-0d95-4472-9d7f-68267d777720, b9357e1e-e7dc-4f0d-97c9-b05adf10e708,
# results/sampler_benchmark-QCTRL
2fa397f9-140d-4d70-a69c-f71cd0cb7983, 6e5c0afc-7d48-4e77-9ab1-537cb790e9fd, c656f3ae-368d-49c1-b2cb-d240e15d9cd3, 60ac4581-8b74-4078-a6b9-1708efd15b40, 4b3e4ab5-0f54-4522-a303-2e51f8af178e, 26c2dee4-b8ba-4be9-933b-b74c43d772de, 7472b14f-7ccf-40a3-9aa6-560f2e9be570,
# results/sampler_benchmark-QCTRL-RND
f64d97a4-2a5e-41f5-8507-4a0f082b1430, d5c8dff5-4fc2-4619-a484-b2230b152882, 9a455734-b445-4984-bf1d-38788a2ed205, 08cce1b7-1caa-4c35-9fa6-a798bd361d08,
# results/tfim_probe
2fd27b98-4cda-4fdb-b34a-392b568b25d9, 16ea48d1-d9d6-403d-b399-adccc7d42997, 2414fb86-2f60-4d78-bfbb-a36c78d26914, 923a7c33-5eea-476b-9794-41da3a5aa471, 4b80b737-f982-4d36-ae9e-cc01ad870cb6, 2a3f7475-37b2-4dbc-a3ec-01391a4c0c25,
# results/tfim_probe-10K-8sites
e78d8b74-4a29-40e7-ad19-d2a4993e6464, cbba293e-5e58-4f36-8f47-2c64e3fe0fe7, fdc2aa2e-a8c8-4681-ac7f-d3d16560b6c2, 7cd1ceca-76ec-4a85-b07a-28d7be62b11f, 6bda7873-b35f-474c-a9e2-be3fef82c862, 9b5a3e56-b669-48a9-bd7d-fbf1fcc8f450, 7d2ed44f-6934-4326-8087-e58d95cdf600, e6890fb9-566d-4f4e-be9b-ea9a6a5f9e86, 683fec4d-6773-4952-9f9e-25ceec279e21, 6e591dc1-f7ab-427b-a8d9-cc674d71e008, a4ce15be-ae59-44a9-b011-4d2780d87c33, a3f21783-ecae-4aa5-b856-9f40f4317e07, aa25a037-ffbc-4b16-afd0-2d8598b6dc61, 32fc35fb-af5d-4a0e-afb6-0fc118fcfc51, 3e465984-7894-4689-a472-2982872b9eac, 665229b6-ebcf-4027-a0c0-07f4298be76a, 93ac77b4-4fe6-47bd-b257-b9444355e6b0, d82c6d70-e03b-40c7-b69f-cd6076d04cca,
# results/tfim_probe-16layers-full-chain
29a732de-9d31-4304-86bc-7b5eb67578bf, e3c41fb7-3dd8-4648-ba29-035cd66270ec, 3a92da7c-e583-4ab4-a207-7ef261f91961, 98a979a2-53bc-4c47-8dcb-3068c66a7d5f, aa62ab44-d7d3-4e61-8408-c76593295f23, e8b36bbc-426c-45db-913f-3e6b9c23b3c6, 8dcc5f45-524f-4973-90f7-aac50f701c64,
"""

BATCH_IDS_TEXT = ""  # No batch/session IDs were present in the result JSONs.
# Testing only (not paper data): BATCH_IDS_TEXT = "5610d686-4042-4f6f-93a6-89d3155a4862"

RUNTIME_JOB_IDS = parse_ids(RUNTIME_JOB_IDS_TEXT)
FUNCTION_JOB_IDS = parse_ids(FUNCTION_JOB_IDS_TEXT)
BATCH_IDS = parse_ids(BATCH_IDS_TEXT)

INCLUDE_BACKEND_PROPERTIES = True  # Identical historical snapshots are stored once per export.
SAVE_JSON = True
OUTPUT_PATH = Path("results/ibm_enriched_job_metadata.json")

if not (RUNTIME_JOB_IDS or FUNCTION_JOB_IDS or BATCH_IDS):
    raise ValueError("Add at least one Runtime, Function, or batch ID.")

print(f"Configured {len(RUNTIME_JOB_IDS)} Runtime jobs, {len(FUNCTION_JOB_IDS)} Function jobs, and {len(BATCH_IDS)} batches.")


Configured 64 Runtime jobs, 50 Function jobs, and 0 batches.


## 2. Authenticate

You can paste the credentials directly into the three string variables below. Any variable left empty is loaded from `.env` instead, so direct notebook values override `.env`. Q-CTRL/QESEM function-job retrieval also requires that the selected IBM instance be entitled to Qiskit Functions; otherwise those errors are recorded while Runtime retrieval continues. Before publishing the notebook, clear any token pasted into the source. Credentials are never written to the output JSON.


In [2]:
import os

from dotenv import load_dotenv
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from qiskit_ibm_runtime import QiskitRuntimeService

# Option 1: paste values here. Leave them empty to use Option 2 (`.env`).
IBM_QUANTUM_TOKEN = ""
IBM_QUANTUM_INSTANCE = ""
IBM_QUANTUM_CHANNEL = ""

# Option 2: load any value that was left empty above.
load_dotenv(".env")
IBM_QUANTUM_TOKEN = (
    IBM_QUANTUM_TOKEN.strip()
    or os.getenv("IBM_QUANTUM_TOKEN")
    or os.getenv("IBM_QUANTUM_TOKEN_EVIDA")
    or ""
)
IBM_QUANTUM_INSTANCE = (
    IBM_QUANTUM_INSTANCE.strip()
    or os.getenv("IBM_QUANTUM_INSTANCE")
    or os.getenv("IBM_QUANTUM_INSTANCE_CUA_EVIDA")
    or os.getenv("IBM_QUANTUM_INTANCE_CUA_EVIDA")  # Older notebook typo.
    or ""
)
IBM_QUANTUM_CHANNEL = (
    IBM_QUANTUM_CHANNEL.strip()
    or os.getenv("IBM_QUANTUM_CHANNEL")
    or "ibm_quantum_platform"
)

if not IBM_QUANTUM_TOKEN or not IBM_QUANTUM_INSTANCE:
    raise ValueError("Both the IBM Quantum token and instance are required.")

service = QiskitRuntimeService(
    channel=IBM_QUANTUM_CHANNEL,
    token=IBM_QUANTUM_TOKEN,
    instance=IBM_QUANTUM_INSTANCE,
)

catalog = None
catalog_auth_error = None
if FUNCTION_JOB_IDS:
    try:
        catalog = QiskitFunctionsCatalog(
            channel=IBM_QUANTUM_CHANNEL,
            token=IBM_QUANTUM_TOKEN,
            instance=IBM_QUANTUM_INSTANCE,
        )
    except Exception as exc:
        catalog_auth_error = f"{type(exc).__name__}: {exc}"
        print("Function catalog unavailable; Runtime jobs will still be retrieved.")

print("Authenticated for read-only retrieval. No job has been submitted.")


qiskit_runtime_service._discover_account:WARNING:2026-09-07 21:34:06,626: Loading account with the given token. A saved account will not be used.


Authenticated for read-only retrieval. No job has been submitted.


## 3. Retrieval helpers

Each Runtime job retains available fields even if another read fails. Calibration lookup uses the execution-start timestamp, or an explicitly labeled submission-time fallback; without a known time it is skipped. Snapshots include IBM's last-update timestamp and are stored once in `calibration_snapshots`. Availability is reported separately from metadata retrieval.

Circuit summaries list physical indices active in submitted Runtime ISA circuits, including measurement and control-flow blocks. Barriers do not count as activity. These indices identify used hardware; they do not reconstruct the original logical-to-physical layout or establish whether layout selection was noise-aware. Missing or undecodable circuits are reported explicitly.


In [3]:
from datetime import datetime, timezone
from collections import Counter
from typing import Any
import hashlib
import json
import math

SENSITIVE_KEYS = ("token", "api_key", "authorization", "instance", "crn", "secret")

def jsonable(value: Any) -> Any:
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, datetime):
        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)
        return value.astimezone(timezone.utc).isoformat()
    if isinstance(value, dict):
        return {
            str(key): "<redacted>" if any(part in str(key).lower() for part in SENSITIVE_KEYS)
            else jsonable(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple, set)):
        return [jsonable(item) for item in value]
    if hasattr(value, "tolist"):
        return jsonable(value.tolist())
    if hasattr(value, "to_dict"):
        return jsonable(value.to_dict())
    return str(value)

def read_member(obj: Any, name: str) -> Any:
    value = getattr(obj, name, None)
    return value() if callable(value) else value

def attempt(errors, field, getter):
    """Keep other metadata when an individual read fails."""
    try:
        return getter()
    except Exception as exc:
        errors[field] = f"{type(exc).__name__}: {exc}"
        return None

def circuit_summary(pub):
    circuit = pub.circuit if hasattr(pub, "circuit") else pub[0]
    active, measured = set(), []
    def visit(block, physical_indices):
        for instruction in block.data:
            indices = [physical_indices[block.find_bit(q).index] for q in instruction.qubits]
            if instruction.operation.name != "barrier":
                active.update(indices)
            if instruction.operation.name == "measure":
                measured.extend(indices)
            for inner in getattr(instruction.operation, "blocks", ()):
                visit(inner, indices)
    visit(circuit, list(range(circuit.num_qubits)))
    return {
        "qubit_index_basis": "submitted Runtime ISA circuit (physical backend indices)",
        "circuit_width": circuit.num_qubits,
        "active_physical_qubits": sorted(active),
        "measured_physical_qubits": sorted(set(measured)),
        "operation_counts": dict(circuit.count_ops()),
    }

def input_summary(job):
    inputs = read_member(job, "inputs")
    if not isinstance(inputs, dict):
        return {"available": False, "reason": "No decoded input dictionary returned."}
    summary = {key: jsonable(value) for key, value in inputs.items() if key != "pubs"}
    pubs = inputs.get("pubs")
    summary["pub_count"] = len(pubs) if pubs is not None else None
    summary["circuits"] = []
    for index, pub in enumerate(pubs if pubs is not None else []):
        errors = {}
        circuit = attempt(errors, "circuit_summary", lambda: circuit_summary(pub))
        summary["circuits"].append({"pub_index": index, "summary": circuit, "errors": errors})
    return summary

def calibration_record(backend, metrics, creation_date):
    record = {
        "status": "disabled" if not INCLUDE_BACKEND_PROPERTIES else "unavailable",
        "lookup_time_utc": None,
        "lookup_time_source": None,
        "snapshot_last_update_utc": None,
        "snapshot_id": None,
        "error": None,
    }
    if not INCLUDE_BACKEND_PROPERTIES:
        return record
    running = ((metrics or {}).get("timestamps") or {}).get("running")
    selected_time = running or creation_date
    record["lookup_time_source"] = "execution_start" if running else (
        "submission_time_fallback" if creation_date else None
    )
    if selected_time is None:
        record["reason"] = "No execution or submission timestamp; historical lookup skipped."
        return record
    try:
        if isinstance(selected_time, str):
            selected_time = datetime.fromisoformat(selected_time.replace("Z", "+00:00"))
        if selected_time.tzinfo is None:
            raise ValueError("Historical lookup timestamp has no timezone.")
        selected_time = selected_time.astimezone(timezone.utc)
        record["lookup_time_utc"] = selected_time.isoformat()
        if backend is None:
            record["reason"] = "Job backend unavailable."
            return record
        # Explicit date selection matches job.properties(), while documenting the fallback.
        properties = backend.properties(datetime=selected_time)
        if properties is None:
            record["reason"] = "IBM returned no historical properties (possibly a retired backend)."
            return record
        snapshot = jsonable(properties)
        record["snapshot_last_update_utc"] = snapshot.get("last_update_date")
        if record["snapshot_last_update_utc"]:
            updated = datetime.fromisoformat(record["snapshot_last_update_utc"].replace("Z", "+00:00"))
            if updated.tzinfo is None or updated > selected_time:
                raise ValueError("Calibration timestamp is undated or newer than the requested historical time.")
        snapshot_id = hashlib.sha256(
            json.dumps(snapshot, sort_keys=True, allow_nan=False).encode()
        ).hexdigest()
        calibration_snapshots[snapshot_id] = snapshot
        record.update(status="retrieved", snapshot_id=snapshot_id)
    except Exception as exc:
        record.update(status="failed", error=f"{type(exc).__name__}: {exc}")
    return record

def retrieve_runtime_job(job, job_id):
    errors = {}
    backend = attempt(errors, "backend", lambda: job.backend(timeout=1))
    fields = {
        "status": "status",
        "creation_date_utc": "creation_date",
        "primitive_id": "primitive_id",
        "session_or_batch_id": "session_id",
        "runtime_image": "image",
        "tags": "tags",
        "metrics": "metrics",
        "usage_estimation": "usage_estimation",
    }
    record = {"job_id": job_id}
    for field, member in fields.items():
        record[field] = attempt(errors, field, lambda member=member: read_member(job, member))
    record["backend_name"] = getattr(backend, "name", None)
    record["inputs_without_circuits"] = attempt(errors, "inputs", lambda: input_summary(job))
    record["backend_calibration"] = calibration_record(
        backend, record["metrics"], record["creation_date_utc"]
    )
    record["field_errors"] = errors
    record["retrieval_status"] = "partial" if errors else "retrieved"
    return jsonable(record)

runtime_record_cache = {}
calibration_snapshots = {}

def retrieve_runtime_job_id(job_id):
    if job_id not in runtime_record_cache:
        try:
            record = retrieve_runtime_job(service.job(job_id), job_id)
        except Exception as exc:
            record = {
                "job_id": job_id,
                "retrieval_status": "failed",
                "retrieval_error": f"{type(exc).__name__}: {exc}",
                "backend_calibration": {"status": "unavailable", "reason": "Job lookup failed."},
            }
        runtime_record_cache[job_id] = record
    return runtime_record_cache[job_id]

def retrieve_function_job(function_job_id):
    errors = {}
    record = {
        "function_job_id": function_job_id, "status": None, "function_metadata": None,
        "runtime_session_ids": [], "runtime_job_ids": [], "runtime_jobs": [],
        "child_discovery_status": "unavailable", "field_errors": errors,
        "retrieval_status": "failed", "retrieval_error": None,
    }
    if catalog is None:
        record["retrieval_error"] = catalog_auth_error or "Qiskit Functions catalog unavailable."
        return record
    job = attempt(errors, "function_lookup", lambda: catalog.job(function_job_id))
    if job is None:
        record["retrieval_error"] = errors.get("function_lookup", "Function job was not found.")
        return record
    record["status"] = attempt(errors, "status", job.status)
    record["function_metadata"] = attempt(errors, "function_metadata", lambda: jsonable(job.raw_data))
    sessions = attempt(errors, "runtime_sessions", job.runtime_sessions) or []
    child_ids = attempt(errors, "runtime_jobs", job.runtime_jobs) or []
    record["runtime_session_ids"] = list(dict.fromkeys(sessions))
    if not child_ids:
        for session_id in record["runtime_session_ids"]:
            children = attempt(
                errors, f"session_jobs:{session_id}",
                lambda session_id=session_id: service.jobs(limit=None, session_id=session_id),
            )
            if children is not None:
                child_ids.extend(str(read_member(child, "job_id")) for child in children)
        if child_ids:
            record["child_discovery_status"] = "recovered_from_sessions"
    else:
        record["child_discovery_status"] = "retrieved"
    record["runtime_job_ids"] = list(dict.fromkeys(child_ids))
    record["runtime_jobs"] = [retrieve_runtime_job_id(jid) for jid in record["runtime_job_ids"]]
    if not child_ids:
        record["child_discovery_status"] = "failed" if errors else "unavailable"
    incomplete_children = not child_ids or any(
        child["retrieval_status"] != "retrieved" for child in record["runtime_jobs"]
    )
    record["retrieval_status"] = "partial" if errors or incomplete_children else "retrieved"
    return jsonable(record)


## 4. Retrieve existing records

Direct IDs use `service.job`; Function IDs use `catalog.job` and its reported Runtime jobs/sessions. When Function child IDs are missing or their lookup fails, reported sessions are expanded with `service.jobs`. Job IDs are deduplicated. Batch details and child-job discovery are attempted independently. Rerunning this cell starts a fresh retrieval and retries previous failures.

All calls retrieve existing records; there are no workload submission or result-waiting calls.


In [4]:
from qiskit_ibm_runtime import Batch

# Reset per export so rerunning this cell retries failures and respects changed settings.
runtime_record_cache.clear()
calibration_snapshots.clear()

runtime_job_records = []
for index, job_id in enumerate(RUNTIME_JOB_IDS, start=1):
    print(f"Runtime job {index}/{len(RUNTIME_JOB_IDS)}: {job_id}")
    runtime_job_records.append(retrieve_runtime_job_id(job_id))

function_job_records = []
for index, job_id in enumerate(FUNCTION_JOB_IDS, start=1):
    print(f"Function job {index}/{len(FUNCTION_JOB_IDS)}: {job_id}")
    function_job_records.append(retrieve_function_job(job_id))

batch_records = []
for batch_id in BATCH_IDS:
    print(f"Batch/session: {batch_id}")
    errors = {}
    batch = attempt(errors, "batch", lambda: Batch.from_id(batch_id, service=service))
    record = {
        "batch_id": batch_id, "details": None, "usage_seconds": None,
        "runtime_job_ids": [], "runtime_jobs": [], "field_errors": errors,
    }
    if batch is not None:
        record["details"] = attempt(errors, "details", lambda: jsonable(batch.details()))
        record["usage_seconds"] = attempt(errors, "usage_seconds", batch.usage)
    jobs = attempt(errors, "runtime_jobs", lambda: service.jobs(limit=None, session_id=batch_id))
    if jobs is not None:
        record["runtime_job_ids"] = list(dict.fromkeys(str(read_member(job, "job_id")) for job in jobs))
        record["runtime_jobs"] = [retrieve_runtime_job_id(jid) for jid in record["runtime_job_ids"]]
    children_failed = any(child["retrieval_status"] != "retrieved" for child in record["runtime_jobs"])
    record["retrieval_status"] = (
        "failed" if batch is None and jobs is None else
        "partial" if errors or children_failed else "retrieved"
    )
    batch_records.append(jsonable(record))

print("Retrieval finished. Coverage and failures are summarized in the next cell.")


Runtime job 1/64: d8qvms6gbcrc73f3v89g
Runtime job 2/64: d8r53deab0ds73dr8phg
Runtime job 3/64: d8rf4dmab0ds73drk25g
Runtime job 4/64: d8ri28i01fac73d4c7v0
Runtime job 5/64: d97cvqd2su3c739hvcf0
Runtime job 6/64: d96vk7sqp3as739r3t1g
Runtime job 7/64: d97f5iaf47jc73a6m8a0
Runtime job 8/64: d96ql9if47jc73a5tqtg
Runtime job 9/64: d96rhjgtcv6s73dk4if0
Runtime job 10/64: d96rta4qp3as739qvb6g
Runtime job 11/64: d979bokqp3as739rg9fg
Runtime job 12/64: d96thj8tcv6s73dk6o7g
Runtime job 13/64: d96uof0tcv6s73dk88hg
Runtime job 14/64: d97bctif47jc73a6i6h0
Runtime job 15/64: d96v222f47jc73a62o8g
Runtime job 16/64: d97eh3if47jc73a6lhcg
Runtime job 17/64: d96qk44qp3as739qu2mg
Runtime job 18/64: d96r6u4qp3as739qukng
Runtime job 19/64: d96rsj4qp3as739qvabg
Runtime job 20/64: d978e4kqp3as739rf8pg
Runtime job 21/64: d96stogtcv6s73dk60h0
Runtime job 22/64: d96tkft2su3c739hcac0
Runtime job 23/64: d9a036gtcv6s73dnrudg
Runtime job 24/64: d9a030l2su3c739l0asg
Runtime job 25/64: d9a032otcv6s73dnru8g
Runtime j

## 5. Review and optionally save one JSON file

The schema-v2 JSON retains direct Runtime jobs, Function jobs and their children, and explicit batches. Each job's `backend_calibration.snapshot_id` references a full entry in `calibration_snapshots`. Calibration states are `retrieved`, `unavailable`, `failed`, or `disabled`; the summary reports this coverage independently of metadata retrieval. A submission-time fallback is not an execution-time calibration claim. Review submitted options and Function metadata before sharing.


In [5]:
all_runtime_records = list(runtime_record_cache.values())
summary = {
    "direct_runtime_job_count": len(runtime_job_records),
    "function_job_count": len(function_job_records),
    "batch_count": len(batch_records),
    "unique_runtime_job_count": len(all_runtime_records),
    "runtime_metadata_status_counts": dict(Counter(item["retrieval_status"] for item in all_runtime_records)),
    "function_metadata_status_counts": dict(Counter(item["retrieval_status"] for item in function_job_records)),
    "function_child_discovery_status_counts": dict(Counter(item["child_discovery_status"] for item in function_job_records)),
    "batch_metadata_status_counts": dict(Counter(item["retrieval_status"] for item in batch_records)),
    "calibration_status_counts": dict(Counter(item["backend_calibration"]["status"] for item in all_runtime_records)),
    "calibration_lookup_source_counts": dict(Counter(
        item["backend_calibration"].get("lookup_time_source") or "none"
        for item in all_runtime_records
    )),
    "unique_calibration_snapshot_count": len(calibration_snapshots),
}
retrieved_payload = jsonable({
    "schema_version": 2,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "read_only": True,
    "include_backend_properties": INCLUDE_BACKEND_PROPERTIES,
    "input_ids": {
        "runtime_job_ids": RUNTIME_JOB_IDS,
        "function_job_ids": FUNCTION_JOB_IDS,
        "batch_ids": BATCH_IDS,
    },
    "summary": summary,
    "catalog_auth_error": catalog_auth_error,
    "runtime_jobs": runtime_job_records,
    "function_jobs": function_job_records,
    "batches": batch_records,
    "calibration_snapshots": calibration_snapshots,
})

display(retrieved_payload["summary"])

if SAVE_JSON:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(retrieved_payload, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write("\n")
    print(f"Saved: {OUTPUT_PATH.resolve()}")
else:
    print("SAVE_JSON=False; nothing was written.")


{'direct_runtime_job_count': 64,
 'function_job_count': 50,
 'batch_count': 0,
 'unique_runtime_job_count': 98,
 'runtime_metadata_status_counts': {'retrieved': 86, 'failed': 12},
 'function_metadata_status_counts': {'retrieved': 34, 'partial': 16},
 'function_child_discovery_status_counts': {'retrieved': 34,
  'unavailable': 16},
 'batch_metadata_status_counts': {},
 'calibration_status_counts': {'retrieved': 86, 'unavailable': 12},
 'calibration_lookup_source_counts': {'execution_start': 86, 'none': 12},
 'unique_calibration_snapshot_count': 40}

Saved: D:\Dropbox\PC\Documents\Documents_Drive\Github-Code\Quantum_Benchmark_26\results\ibm_enriched_job_metadata.json
